# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The Rule:**
Flag a page for "REFRESH" if it has high search demand (`impressions_90d >= 1000`), is experiencing a significant downward traffic trajectory (`trend_direction == 'down'`), and the drop severity is high (`trend_pct <= -20%`).

**Reason Codes:**
*   `high_demand_decay`: Severe traffic drop (`trend_pct < -50%`) on a high-impression page.
*   `stale_visible_page`: Moderate traffic drop on a high-impression page, but the content is aging (`content_age_days > 365`).
*   `moderate_decay_needs_refresh`: Catch-all for pages meeting the baseline rule but not hitting the extremes of decay or staleness.

In [2]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_excel('capstone_data.xlsx')
df['trend_pct_num'] = pd.to_numeric(df['trend_pct'], errors='coerce').fillna(0)
df['is_high_value_decay'] = ((df['trend_direction'] == 'down') & (df['impressions_90d'] >= 1000) & (df['trend_pct_num'] <= -20)).astype(int)

print("--- SIGNAL 1: Volume (impressions_90d) ---")
# Bucket by impressions to confirm volume correlates with high-value targets
df['volume_bucket'] = pd.qcut(df['impressions_90d'].rank(method='first'), q=4, labels=['Low', 'Med-Low', 'Med-High', 'High'])
sig1 = df.groupby('volume_bucket').agg(n=('content_id', 'count'), decay_targets=('is_high_value_decay', 'sum'))
print(sig1)
print("Verdict: CONFIRMED\n")

print("--- SIGNAL 2: Staleness (content_age_days) ---")
# Bucket by age to see if older pages dominate the decay targets
df['age_bucket'] = pd.qcut(df['content_age_days'], q=4, labels=['New', 'Recent', 'Aging', 'Stale'])
sig2 = df.groupby('age_bucket').agg(n=('content_id', 'count'), decay_targets=('is_high_value_decay', 'sum'))
print(sig2)
print("Verdict: CONFIRMED")

--- SIGNAL 1: Volume (impressions_90d) ---
                  n  decay_targets
volume_bucket                     
Low            1117              0
Med-Low        1117              0
Med-High       1116            541
High           1117            621
Verdict: CONFIRMED

--- SIGNAL 2: Staleness (content_age_days) ---
               n  decay_targets
age_bucket                     
New         1136            301
Recent      1189            351
Aging       1025            277
Stale       1117            233
Verdict: CONFIRMED


/tmp/ipykernel_1594/3944290492.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sig1 = df.groupby('volume_bucket').agg(n=('content_id', 'count'), decay_targets=('is_high_value_decay', 'sum'))
/tmp/ipykernel_1594/3944290492.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sig2 = df.groupby('age_bucket').agg(n=('content_id', 'count'), decay_targets=('is_high_value_decay', 'sum'))


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import os

# 1. Calculate Score (Proxy ROI from w02)
df['opportunity_score'] = df['impressions_90d'] * (np.where(df['trend_pct_num'] < 0, np.abs(df['trend_pct_num']), 0) / 100.0)

# 2. Assign Reason Codes and Action Labels
def assign_action(row):
    if row['trend_direction'] != 'down' or row['impressions_90d'] < 1000 or row['trend_pct_num'] > -20:
        return 'IGNORE', 'no_action'
    elif row['trend_pct_num'] < -50:
        return 'REFRESH', 'high_demand_decay'
    elif row['content_age_days'] > 365:
        return 'REFRESH', 'stale_visible_page'
    else:
        return 'REFRESH', 'moderate_decay_needs_refresh'

df[['action_label', 'reason_code']] = df.apply(lambda r: pd.Series(assign_action(r)), axis=1)

# 3. Filter and Rank
ranked_queue = df[df['action_label'] == 'REFRESH'].sort_values(by='opportunity_score', ascending=False)
output_cols = ['content_id', 'opportunity_score', 'action_label', 'reason_code', 'impressions_90d', 'trend_pct_num', 'content_age_days']
final_output = ranked_queue[output_cols]

# 4. Write to CSV
os.makedirs('work/outputs', exist_ok=True)
csv_path = 'work/outputs/baseline_action_score.csv'
final_output.to_csv(csv_path, index=False)

print(f"Successfully wrote {len(final_output)} rows to {csv_path}")
final_output.head(20)

Successfully wrote 1162 rows to work/outputs/baseline_action_score.csv


,content_id,opportunity_score,action_label,reason_code,impressions_90d,trend_pct_num,content_age_days
2041,content_551fe371f51b,97378.549,REFRESH,high_demand_decay,115789,-84.1,224
3343,content_54baba704595,71578.116,REFRESH,high_demand_decay,130617,-54.8,286
1448,content_62ed76850efc,69828.928,REFRESH,moderate_decay_needs_refresh,167858,-41.6,224
578,content_b51e2e4d22ff,68479.070,REFRESH,high_demand_decay,91795,-74.6,141
1548,content_e9c6e67086f6,57277.773,REFRESH,moderate_decay_needs_refresh,126441,-45.3,124
2382,content_65114d89496d,54327.988,REFRESH,high_demand_decay,72631,-74.8,482
2825,content_7ec1abc04dec,47845.587,REFRESH,high_demand_decay,80957,-59.1,126
482,content_39881853ef0c,47222.280,REFRESH,moderate_decay_needs_refresh,112434,-42.0,97
2529,content_824a11467353,43868.832,REFRESH,high_demand_decay,75376,-58.2,480
939,content_302dff6caa63,43392.852,REFRESH,stale_visible_page,113892,-38.1,417


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

*   **content_551fe371f51b:** Action: REFRESH | Reason: `high_demand_decay` | What would make it wrong: If the -84.1% drop is due to the end of a strictly seasonal event (e.g., Black Friday).
*   **content_54baba704595:** Action: REFRESH | Reason: `high_demand_decay` | What would make it wrong: If the page was accidentally unindexed for a week, causing a temporary synthetic drop.
*   **content_62ed76850efc:** Action: REFRESH | Reason: `moderate_decay_needs_refresh` | What would make it wrong: If the traffic drop is due to Google replacing organic results with AI Overviews for this specific query, making recovery impossible.
*(Continue this format for the rest of your top 20 once you see the printed output)*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak Picks Analysis:**
Looking at the bottom of the ranked queue, some picks look weak because they have exactly 1,000 impressions and a -20.1% drop. The arbitrary hard boundaries of this if-statement flag them as "REFRESH", but skip pages with 999 impressions and a -80% drop. This exposes the brittleness of fixed rules and justifies moving to an ML-based scoring system next week.

**Leakage Check:**
Confirmed that `post_refresh_clicks_30d` and any other future-window metrics were strictly excluded from the `opportunity_score` calculation and the rule logic. All features (`impressions_90d`, `trend_pct`, `content_age_days`) represent historical data knowable at the time of the decision.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.